### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [1]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = True

### Start with our Message class

In [2]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [3]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [4]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [5]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen in 2026.  Take into account the \
uptake and future investment Microsoft is making into the product; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen in 2026.  Take into account the \
uptake and future investment Microsoft is making into the product; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [6]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [7]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [8]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [9]:
display(Markdown(response.content))

## Pros of AutoGen:
In considering AutoGen for an AI Agent project in 2026, several compelling reasons support its adoption:

1. **Strong Support from Microsoft**: Microsoft has shown a significant commitment to the AutoGen framework, with considerable investment in its development and integration with other tools. This backing enhances its credibility and ensures continued evolution and support from a major player in the AI space.

2. **Multi-Agent Systems**: AutoGen excels in building multi-agent systems, allowing different agents to collaborate and perform complex tasks more efficiently than single-agent systems. This capability opens up new possibilities for automation and efficiency in various applications.

3. **Flexibility and Orchestration**: The conversation-based orchestration approach used by AutoGen enables dynamic interaction between agents. This flexibility can lead to more natural and efficient problem-solving capabilities, as agents can adapt to changing contexts and inputs in real time.

4. **Open-Source Community**: As an open-source framework, AutoGen benefits from community contributions, ensuring that it remains robust, innovative, and continuously improved. This can lead to faster iteration, sharing of best practices, and the incorporation of cutting-edge techniques from the AI community.

5. **Proven Impact**: There are real-world case studies demonstrating significant improvements in development efficiency, model performance, and cost reduction when using AutoGen. Organizations that have adopted AutoGen report better outcomes in their AI initiatives.

6. **Integration Potential**: The potential for integration with other Microsoft products and tools adds to AutoGen's appeal, providing users with a more cohesive and powerful set of resources for their AI projects. 

In summary, the future investment from Microsoft, combined with the versatile and efficient features of AutoGen, makes it a strong candidate for AI agent development in 2026.

TERMINATE

## Cons of AutoGen:
In considering whether to use AutoGen for an AI Agent project in 2026, there are several reasons against selecting this framework:

1. **Steep Learning Curve**: AutoGen is complex and tailored for technical profiles, which may necessitate significant training and ramp-up time for teams unfamiliar with its intricacies.

2. **Over-engineering for Simple Use Cases**: For straightforward applications, the capabilities offered by AutoGen may be excessive, leading to unnecessary complexity in development.

3. **High Development and Maintenance Effort**: While AutoGen can excel in advanced scenarios, it often requires hands-on coding and cannot provide the out-of-the-box functionalities offered by other frameworks. This can result in increased time and resource investments.

4. **Limited Integration Features**: The framework reportedly lacks built-in authentication and role-based access control (RBAC) features, and has limited native integrations with business applications, which could complicate deployment in business environments.

5. **Focus on Prototyping Rather Than Production**: While AutoGen is adept for prototyping, transitioning a project to production can be challenging and may require additional development to resolve its limitations.

6. **Potential Isolation Due to Future Updates**: As Microsoft continues to invest in AutoGen, there is a possibility that it may diverge from other platforms, creating challenges in interoperability and compatibility with existing or future tools.

7. **Alternative Solutions**: Competing frameworks may offer more refined features, better integration, and easier usability that could be more beneficial for tailored applications.

In summary, while Microsoft’s backing of AutoGen may provide it with robust scenarios and development support, the complexities and limitations of the system raise valid concerns for its use in new projects. 

TERMINATE



## Decision:

After reviewing the comprehensive pros and cons presented by the research team, my decision is to **not use AutoGen for the project**. 

The compelling advantages, such as strong support from Microsoft and its capabilities for building multi-agent systems, are significant. However, the steep learning curve, high development and maintenance effort required, and the possibility of over-engineering for simpler use cases outweigh these benefits. Additionally, the potential challenges regarding transitioning to production and limited integration features present considerable risks for our project. 

Given these factors, I recommend considering alternative solutions that may offer a more balanced approach to usability, integration, and effective deployment for our specific needs. 

TERMINATE

In [10]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [11]:
await host.stop()